# NumPy Vectorization

## Fokus Bab

Bab ini membahas pergeseran cara berpikir paling fundamental saat bekerja dengan NumPy: dari loop eksplisit menjadi operasi array (vectorized operation). Vectorization bukan sekadar gaya penulisan kode yang lebih ringkas, melainkan mekanisme di balik alasan mengapa NumPy dapat memproses data numerik berskala besar jauh lebih cepat dibanding Python murni.

## Tujuan Pembelajaran

* Menjelaskan konsep vectorization dan membedakannya dari pendekatan loop konvensional.
* Mengonversi kode berbasis loop Python menjadi vectorized operation menggunakan NumPy.
* Menerapkan conditional vectorization menggunakan np.where(), np.clip(), dan operasi boolean untuk menghindari if-else di dalam loop.
* Mengukur dan membandingkan performa kode menggunakan time, timeit, dan decorator kustom @timer.
* Menjelaskan secara teknis mengapa NumPy jauh lebih cepat dibanding loop Python murni.

## Goals

Mampu mengganti operasi numerik berbasis loop dengan vectorized operation dan membandingkan performanya secara empiris menggunakan time, timeit, atau decorator @timer. Kemampuan ini menjadi dasar praktik terbaik penulisan kode NumPy yang efisien, sekaligus menjelaskan mengapa operasi array dan broadcasting (bab 4) selalu diprioritaskan dibanding loop eksplisit.

## Konsep Vectorization

Vectorization: eknik melakukan operasi numerik pada array secara langsung sebagai satu kesatuan, sehingga operasi diterapkan ke setiap elemen tanpa menulis loop Python secara eksplisit.

Beberapa poin untuk memahami konsep ini:

* Vectorized operation adalah operasi yang diterapkan pada array secara langsung, tanpa loop Python eksplisit pada setiap elemen.
* Vectorization tidak berarti selalu satu instruksi CPU untuk seluruh array. Implementasi internal tetap dapat melakukan iterasi terhadap elemen, tetapi iterasi tersebut dijalankan oleh kode tingkat rendah NumPy/C dan dapat memanfaatkan optimasi seperti SIMD.
* Tujuan utama vectorization adalah mengurangi overhead loop Python dan memanfaatkan implementasi numerik yang lebih efisien.
* Banyak operasi NumPy sudah bersifat vectorized, misalnya:

  ```python
  arr + 10
  np.sqrt(arr)
  np.sum(arr)
  ```
* `np.sum()` lebih tepat disebut operasi reduction, karena menggabungkan banyak elemen menjadi satu nilai.
* Perubahan cara berpikir saat menggunakan NumPy:

  > Jangan bertanya "bagaimana saya mengulang setiap elemen?", tetapi tanyakan "operasi array apa yang dapat menggantikan loop ini?"

Inti:

> Vectorization adalah teknik melakukan operasi langsung pada array dengan memanfaatkan implementasi numerik yang efisien di level rendah, sehingga kita tidak perlu menulis loop Python per elemen.

## Loop vs NumPy

membandingkan langsung dua pendekatan untuk menyelesaikan permasalahan yang sama: mengalikan setiap elemen array dengan 2.

### Pendekatan loop (Python murni):

In [5]:
import numpy as np

In [8]:
x = np.array([1, 2, 3, 4, 5])

for i in range(len(x)):
    x[i] *=2
print(x)

[ 2  4  6  8 10]


Penjelasan kode di atas:

* Python menjalankan `range(len(x))` untuk menghasilkan indeks `0, 1, 2, 3, 4`, kemudian melakukan iterasi satu per satu.
* Pada setiap iterasi, Python mengambil elemen `x[i]`, melakukan operasi perkalian, lalu menyimpan hasilnya kembali ke posisi tersebut.
* Setiap langkah melibatkan proses tambahan di interpreter Python, seperti pengelolaan objek dan pemanggilan operasi.
* Biaya tambahan yang muncul dari proses tersebut disebut interpreter overhead.
* Interpreter overhead tidak berasal dari operasi perkalian itu sendiri, tetapi dari proses yang diperlukan Python untuk menjalankan loop dan operasinya pada setiap iterasi.
* Semakin banyak elemen yang diproses, semakin sering overhead tersebut terjadi.


### Pendekatan vectorized (NumPy):

In [11]:
x = np.array([1, 2, 3, 4, 5])

x *= 2
print(x)

[ 2  4  6  8 10]


Penjelasan kode di atas:

* Satu baris operasi NumPy dapat menggantikan seluruh loop Python dan menghasilkan output yang sama: `[2, 4, 6, 8, 10]`.
* Alih-alih menulis loop eksplisit di Python, operasi array didelegasikan ke implementasi internal NumPy yang telah dikompilasi dan dioptimalkan.
* NumPy tetap melakukan pemrosesan terhadap setiap elemen, tetapi proses tersebut berlangsung di level rendah, bukan melalui loop Python yang eksplisit.
* Karena overhead loop Python lebih rendah, vectorization umumnya memberikan performa lebih baik, terutama pada array berukuran besar.
* Perbedaan performa kedua pendekatan akan diuji secara empiris pada subbab 1.5 — Benchmark.

## Conditional Vectorization

* Vectorization tidak hanya digunakan untuk operasi aritmatika, tetapi juga untuk logika kondisional.
* NumPy menyediakan fungsi seperti `np.where()` untuk menerapkan kondisi pada seluruh array tanpa `if-else` di dalam loop.
* Intinya: logika yang biasanya ditulis sebagai `if-else` per elemen dapat diubah menjadi operasi array.


`np.where()` — mengganti pola `if-else` per elemen:

In [12]:
arr = np.array([3, -7, 12, -4, 9])
hasil = np.where(arr > 0, arr, 0)
print(hasil)

[ 3  0 12  0  9]


Penjelasan kode di atas:

* `np.where()` menerapkan kondisi `x > 0` pada seluruh elemen array tanpa `if-else` dalam loop Python.
* Jika kondisi `x > 0` bernilai `True`, nilai asli dipertahankan; jika `False`, nilai diganti `0`.
* Hasilnya: `[3, 0, 12, 0, 9]`.
* Secara logika, ini setara dengan `if-else` pada setiap elemen, tetapi ditulis sebagai operasi array.
* Dalam konteks vectorization, `np.where()` digunakan sebagai alat untuk menerapkan logika kondisional secara vectorized.
* Pembahasan Boolean Masking akan dibahas lebih lanjut pada Bab 3.


`np.clip()` — membatasi nilai dalam rentang tertentu:

In [13]:
ararr = np.array([-5, 3, 12, 8, -2, 20])
hasil = np.clip(arr, 0, 10)
print(hasil)

[ 3  0 10  0  9]


Penjelasan kode di atas:

* `np.clip(arr, batas_bawah, batas_atas)` membatasi setiap nilai array agar berada dalam rentang `batas_bawah` hingga `batas_atas`.
* Nilai di bawah `batas_bawah` diubah menjadi `batas_bawah`, sedangkan nilai di atas `batas_atas` diubah menjadi `batas_atas`.
* Hasilnya: `[0, 3, 10, 8, 0, 10]`.
* `np.clip()` melakukan proses tersebut secara vectorized tanpa loop Python eksplisit.
* Dalam praktik Data Science, `np.clip()` dapat digunakan untuk:

  * membatasi probabilitas pada rentang `0–1`,
  * melakukan capping pada nilai ekstrem,
  * menjaga nilai piksel gambar pada rentang `0–255`.
* Catatan: istilah yang lebih tepat untuk `np.clip()` adalah **capping/bounding**, bukan pembulatan.

Boolean operations sebagai vectorized counting:

In [14]:
arr = np.array([3, -7, 12, -4, 9, -1, 8])
jumlah_positif = (arr > 10).sum()
print(jumlah_positif)

1


Penjelasan kode di atas:

* `arr > 0` menghasilkan boolean array: `[True, False, True, False, True, False, True]`.
* Pada `.sum()`, NumPy memperlakukan `True` sebagai `1` dan `False` sebagai `0`.
* Hasil `4` menunjukkan bahwa terdapat **4 elemen yang memenuhi kondisi `arr > 0`**.
* Cara ini menghitung jumlah elemen yang memenuhi kondisi tanpa loop dan counter manual.


## Benchmark

### Modul time:

In [ ]:
import time

start = time.time()

# ...kolom yang ingin diukur...
end = time.time()

print(f"Waktu eksekusi {end - start:.6f} detik")

Penjelasan kode di atas:

* `time.time()` mengembalikan timestamp saat ini dalam satuan detik.
* `end - start` menghitung perkiraan durasi eksekusi kode di antara keduanya.
* Hasil pengukuran dapat dipengaruhi aktivitas lain pada sistem, sehingga kurang stabil.
* Pengukuran hanya dilakukan satu kali, sehingga hasilnya dapat kurang representatif.
* Untuk benchmark yang lebih akurat, gunakan `timeit` karena dapat menjalankan kode berulang kali dan membantu memperoleh pengukuran yang lebih konsisten.


### Modul `timeit` 

lebih presisi, direkomendasikan untuk benchmark serius

In [16]:
import timeit

waktu = timeit.timeit(
    stmt = 'x *= 2',
    setup='import numpy as np; x = np.arange(100000)',
    number=100
)

print(f"Rata rata: {waktu / 100:.8f} detik pereksekusi")

Rata rata: 0.00116197 detik pereksekusi


Penjelasan kode di atas:

* `setup` dijalankan sekali untuk menyiapkan data dan tidak termasuk dalam waktu yang diukur.
* `stmt` berisi kode yang benar-benar diukur.
* `number=100` berarti `stmt` dijalankan sebanyak 100 kali.
* `timeit` mengembalikan total waktu dari seluruh pengulangan tersebut.
* Pengulangan berkali-kali membuat hasil benchmark lebih representatif dan mengurangi pengaruh gangguan sesaat dari sistem.
* Di Jupyter Notebook, `%timeit kode_anda` merupakan cara praktis untuk melakukan benchmark secara otomatis dan menampilkan statistik waktu eksekusi.


### Decorator kustom `@timer`

In [18]:
import time
from functools import wraps


def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        hasil = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} selesai dalam {end - start:.6f} detik")
        return hasil
    return wrapper

@timer
def kalikan_dengan_loop(x):
    for i in range(len(x)):
        x[i] *= 2
    return x


Penjelasan kode di atas:

* `timer` adalah decorator yang menerima fungsi `func`, lalu membungkusnya dengan fungsi `wrapper` untuk menambahkan pengukuran waktu eksekusi.
* `@timer` membuat setiap pemanggilan `kalikan_dengan_loop(...)` otomatis melewati `timer`, sehingga kode pengukuran waktu tidak perlu ditulis berulang.
* `@wraps(func)` menjaga metadata fungsi asli, seperti `__name__`, agar tetap `kalikan_dengan_loop`, bukan berubah menjadi `wrapper`.
* Decorator cocok digunakan untuk benchmark beberapa fungsi karena logika pengukuran waktu dapat digunakan kembali secara konsisten.
* Intinya:

  > Decorator memungkinkan kita menambahkan logika tambahan pada fungsi tanpa mengubah kode utama fungsi tersebut.


## Kenapa NumPy Cepat?

![...](../assets/figures/numpy_loop_vs_vectorized.png)

* Loop Python murni menanggung overhead interpreter pada setiap iterasi karena setiap elemen diproses melalui loop Python.
* Pada operasi vectorized NumPy, pemanggilan operasi dilakukan sekali dari Python, lalu pemrosesan elemen dilakukan oleh implementasi internal NumPy di level rendah.
* Jadi, overhead dari pemanggilan Python tidak berulang untuk setiap elemen.
* Intinya:

  > Loop Python → overhead interpreter berulang per iterasi.
  > NumPy vectorization → overhead pemanggilan Python relatif sekali, lalu operasi array diproses di level rendah.


## 📖 Deep Dive — Tiga Alasan Teknis di Balik Kecepatan NumPy

1. Compiled implementation

   * Operasi inti NumPy banyak diimplementasikan dalam C/C++ dan dikompilasi menjadi kode mesin.
   * Vectorization memanfaatkan implementasi tersebut untuk memproses array tanpa loop Python eksplisit.
   * Akibatnya, operasi dapat berjalan lebih efisien dibandingkan loop yang dieksekusi oleh interpreter Python.

2. Mengurangi Python loop overhead

   * Pada loop Python, setiap iterasi melibatkan overhead interpreter, seperti pengelolaan objek dan pemanggilan operasi.
   * Pada operasi vectorized NumPy, loop pemrosesan dipindahkan ke implementasi internal NumPy di level rendah.
   * Akibatnya, overhead Python tidak perlu terjadi berulang pada setiap elemen.

3. Pemanfaatan SIMD

   * NumPy dan library numerik yang digunakannya dapat memanfaatkan SIMD pada CPU tertentu.
   * SIMD memungkinkan beberapa elemen diproses dalam satu instruksi mesin.
   * Namun, penggunaan SIMD bergantung pada operasi, versi NumPy, library yang digunakan, dan kemampuan CPU; jadi tidak semua operasi NumPy otomatis menggunakan SIMD.

### Inti

> Kecepatan NumPy terutama berasal dari pemrosesan di level rendah yang terkompilasi, berkurangnya overhead loop Python, dan pada kondisi tertentu pemanfaatan optimasi CPU seperti SIMD.

* Secara empiris, operasi vectorized NumPy umumnya lebih cepat daripada loop Python, terutama ketika memproses array berukuran besar.
* Besarnya peningkatan performa bergantung pada jenis operasi, ukuran data, dan lingkungan eksekusi.


## Ringkasan

| Konsep                    | Sintaks Kunci                | Kegunaan Utama                                         |
| ------------------------- | ---------------------------- | ------------------------------------------------------ |
| Vectorized operation      | `x *= 2` (tanpa loop)        | Menggantikan loop elemen-per-elemen                    |
| Conditional vectorization | `np.where()`, `np.clip()`    | Menerapkan logika kondisional tanpa `if-else` manual   |
| Vectorized counting       | `(arr > kondisi).sum()`      | Menghitung jumlah elemen yang memenuhi kondisi         |
| Benchmark manual          | `time.time()`                | Pengukuran cepat, tetapi kurang presisi                |
| Benchmark presisi         | `timeit.timeit()`, `%timeit` | Mengukur performa berdasarkan banyak pengulangan       |
| Benchmark reusable        | Decorator `@timer`           | Mengukur waktu eksekusi banyak fungsi secara konsisten |
